# AIoT Project

In [ ]:
# Standard library imports
from collections import Counter
from datetime import datetime
import os
from time import time

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymongo
from psynlig import pca_explained_variance_bar
import scipy
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import class_weight
import tensorflow as tf
from tensorflow.keras import Sequential, regularizers
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    Dense,
    Dropout,
    Flatten,
    MaxPooling1D,
    LSTM,
    Bidirectional,
    GlobalAveragePooling1D,
    BatchNormalization,
    GaussianNoise
)
from tqdm.notebook import tqdm_notebook
import yaml

# Local application imports
import utils
from utils import encode_labels
import utils_visual

%matplotlib inline
%load_ext autoreload
%autoreload 2

# Start time of execution

## Execution Time Tracking

We'll track the total execution time of this notebook to measure performance and computational efficiency. This helps us evaluate how long the entire pipeline takes to run, which is important for real-time applications.

In [ ]:
time_start = time()

## Load Configuration from YAML

We'll load project settings from a YAML configuration file, which contains:
- Database connection parameters
- Sensor sampling frequencies
- Signal processing parameters
- Model configuration settings

In [ ]:
# Define configuration file path
config_path = os.path.join(os.getcwd(), "config.yml")

# Load configuration settings from YAML file
with open(config_path) as file:
    config = yaml.load(file, Loader=yaml.FullLoader)

# Set up MongoDB connection using configuration
client = pymongo.MongoClient(config["client"])
db = client[config["db"]]
coll = db[config["col"]]

# Print available class labels in the database
found_keys = coll.distinct("label")
print("Available activity classes in database:", found_keys)

## Data Loading from MongoDB

In this section, we'll:
1. Query the MongoDB collection for each activity class
2. Extract IMU sensor data (accelerometer and gyroscope readings)
3. Convert each document to a pandas DataFrame
4. Organize DataFrames by activity class for further processing

In [ ]:
# Get all distinct activity classes
labels = coll.distinct("label")

# Dictionary to store DataFrames by activity class
dfs_by_label = {}

for label in labels:
    cursor = coll.find({"label": label})
    dfs = []
    
    for doc in cursor:
        # Convert sensor data to DataFrame
        df = pd.DataFrame({
            'acc_x': doc['data']['acc_x'],
            'acc_y': doc['data']['acc_y'],
            'acc_z': doc['data']['acc_z'],
            'gyro_x': doc['data']['gyro_x'],
            'gyro_y': doc['data']['gyro_y'],
            'gyro_z': doc['data']['gyro_z']
        })
        dfs.append(df)
    
    dfs_by_label[label] = dfs

# At this point, dfs_by_label contains all sensor data organized by activity class

## Exploratory Data Analysis and Filtering

#### Time-length of samples per class

In [ ]:
total_times = {}
sampling_rate = config["wearable_sample_freq"]

for label, dfs in dfs_by_label.items():
    total_samples = sum(len(df) for df in dfs)
    total_times[label] = total_samples / sampling_rate

plt.figure(figsize=(8, 5))
sns.barplot(x=list(total_times.keys()), y=list(total_times.values()))
plt.ylabel('Total Time (seconds)')
plt.xlabel('Class')
plt.title('Total Time of Data for Each Class')
plt.show()

#### Visualize a Single Document from Each Class

In [ ]:
for label, dfs in dfs_by_label.items():
    df = dfs[0]
    n_samples = len(df)
    sampling_rate = config["wearable_sample_freq"]
    time_axis = np.arange(n_samples) / sampling_rate

    fig, axs = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    # Accelerometer subplot
    axs[0].plot(df['acc_x'], label='acc_x')
    axs[0].plot(df['acc_y'], label='acc_y')
    axs[0].plot(df['acc_z'], label='acc_z')
    axs[0].set_title(f'Accelerometer Signals - Class: {label}')
    axs[0].set_ylabel('Acceleration (g)')
    axs[0].legend()

    # Gyroscope subplot
    axs[1].plot(df['gyro_x'], label='gyro_x')
    axs[1].plot(df['gyro_y'], label='gyro_y')
    axs[1].plot(df['gyro_z'], label='gyro_z')
    axs[1].set_title(f'Gyroscope Signals - Class: {label}')
    axs[1].set_xlabel('Sample')
    axs[1].set_ylabel('Angular Velocity (deg/s)')
    axs[1].legend()

    # Add time in seconds as secondary x-axis
    def samples_to_time(x): return x / sampling_rate
    def time_to_samples(x): return x * sampling_rate

    secax = axs[1].secondary_xaxis('top', functions=(samples_to_time, time_to_samples))
    secax.set_xlabel('Time (seconds)')

    plt.tight_layout()
    plt.show()

#### IMU Data Transformation to World Coordinates

Converting sensor data from device-relative to world-relative coordinates makes the activity recognition model robust to device orientation. This transformation:

- Reduces the model's sensitivity to how the device is worn or positioned
- Helps generalize across different users and wearing positions
- Maintains the characteristics of the activity patterns regardless of orientation

In [ ]:
# Apply world-frame transformation to the full samples (per class)
dfs_world_by_label = {}
for label, dfs in dfs_by_label.items():
    dfs_world = []
    for df in dfs:
        try:
            acc = df[["acc_x", "acc_y", "acc_z"]].values.astype(np.float32)
            gyr = df[["gyro_x", "gyro_y", "gyro_z"]].values.astype(np.float32)
            acc_world = utils.transform_imu_to_world(acc, gyr, rate=sampling_rate)
            df_world = df.copy()
            df_world["acc_x_world"] = acc_world[:, 0]
            df_world["acc_y_world"] = acc_world[:, 1]
            df_world["acc_z_world"] = acc_world[:, 2]
            dfs_world.append(df_world)
        except Exception as e:
            print(f"Error transforming data for label {label}: {e}")
            # Fall back to using original data (without world transformation)
            df_world = df.copy()
            df_world["acc_x_world"] = df["acc_x"].values  # Use original as fallback
            df_world["acc_y_world"] = df["acc_y"].values
            df_world["acc_z_world"] = df["acc_z"].values
            dfs_world.append(df_world)
            
    dfs_world_by_label[label] = dfs_world

# Optionally, check transformation quality for a random sample
try:
    df_sample = next(iter(dfs_world_by_label.values()))[2]
    metrics = utils.check_transformation_quality(df_sample[["acc_x_world", "acc_y_world", "acc_z_world"]].values)
    print(metrics)
except Exception as e:
    print(f"Error checking transformation quality: {e}")

# After processing all data
for label, dfs in dfs_world_by_label.items():
    print(f"\nQuality metrics for {label}:")
    for i, df in enumerate(dfs[:3]):  # Check first 3 samples
        metrics = utils.check_transformation_quality(
            df[["acc_x_world", "acc_y_world", "acc_z_world"]].values
        )
        print(f"Sample {i}: {metrics}")

#### Data Windowing

We'll segment the continuous sensor data into fixed-length windows for machine learning input preparation. For each window, we'll compare the original and world-transformed accelerometer data.

##### Windowing Strategy
- **Window Size**: 1.2 seconds (120 samples at 100Hz)
- **Step Size**: 50% overlap between consecutive windows
- **Benefits**: 
  - Creates standardized input dimensions for machine learning models
  - Overlapping windows prevent missing important transitions between activities
  - Multiple windows per activity instance improves training data diversity

##### Visualization Approach
For each activity class, we'll compare:
1. Original accelerometer data (device-relative)
2. World-transformed accelerometer data (gravity-aligned)
3. Gyroscope data for rotational context

This comparison demonstrates how world-transformation reduces the dependency on device orientation while preserving the characteristic movement patterns of each activity.

In [ ]:
# Define window parameters based on sampling rate

window_size = int(config["sliding_window"]["ws"])  # Window size in samples
step_size = int(window_size * config["sliding_window"]["overlap"])  # 50% overlap

# Create dictionaries to store windowed data
windowed_counts = {}
windowed_samples_by_label = {}

# Process each activity class - use dfs_world_by_label which contains both original and world-transformed data
for label, dfs_world in dfs_world_by_label.items():
    windowed_samples = []
    
    # Process each recording with world-transformed data
    for df_world in dfs_world:
        # Create windows from the DataFrame that contains both original and world-transformed data
        windows = utils.sliding_window_pd(df_world, window_size, step_size)
        windowed_samples.extend(windows)
    
    windowed_counts[label] = len(windowed_samples)
    windowed_samples_by_label[label] = windowed_samples

# Plot the number of windows per class
plt.figure(figsize=(10, 6))
sns.barplot(x=list(windowed_counts.keys()), y=list(windowed_counts.values()))
plt.ylabel('Number of Windows')
plt.xlabel('Activity Class')
plt.title('Window Count Distribution by Activity Class')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize windowed samples for each class, comparing original vs world-transformed data
for label, windows in windowed_samples_by_label.items():
    if not windows:
        continue
        
    # Get a sample window that contains both original and world-transformed data
    sample_window = windows[0]
    n_samples = len(sample_window)
    time_axis = np.arange(n_samples) / sampling_rate
    
    # Create figure with three subplots
    fig, axs = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    # Plot original accelerometer data
    axs[0].plot(time_axis, sample_window['acc_x'], label='acc_x')
    axs[0].plot(time_axis, sample_window['acc_y'], label='acc_y')
    axs[0].plot(time_axis, sample_window['acc_z'], label='acc_z')
    axs[0].set_title(f'Original Accelerometer Signals - Class: {label}')
    axs[0].set_ylabel('Acceleration (g)')
    axs[0].legend()
    
    # Plot world-transformed accelerometer data
    axs[1].plot(time_axis, sample_window['acc_x_world'], label='acc_x_world')
    axs[1].plot(time_axis, sample_window['acc_y_world'], label='acc_y_world')
    axs[1].plot(time_axis, sample_window['acc_z_world'], label='acc_z_world')
    axs[1].set_title(f'World-Transformed Accelerometer Signals - Class: {label}')
    axs[1].set_ylabel('Acceleration (g)')
    axs[1].legend()
    
    # Plot gyroscope data
    axs[2].plot(time_axis, sample_window['gyro_x'], label='gyro_x')
    axs[2].plot(time_axis, sample_window['gyro_y'], label='gyro_y')
    axs[2].plot(time_axis, sample_window['gyro_z'], label='gyro_z')
    axs[2].set_title(f'Gyroscope Signals - Class: {label}')
    axs[2].set_xlabel('Time (seconds)')
    axs[2].set_ylabel('Angular Velocity (deg/s)')
    axs[2].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Optional: Break after the first few classes to avoid too many plots
    # Uncomment the following lines to limit the number of plots
    # if list(windowed_samples_by_label.keys()).index(label) >= 2:
    #     break

#### Signal Filtering

We'll apply a digital filter to each windowed segment of the sensor data. Filtering helps remove noise and unwanted frequency components, improving the quality of features for downstream machine learning.

- **wn (normalized frequency)**: Computed as `2 * (freq / sampling_rate)`
- Filtering is applied to each axis of both accelerometer and gyroscope data.
- For large datasets, this process can be parallelized using `multiprocessing.Pool`, `concurrent.futures`, or libraries like Dask.

In [ ]:
# Retrieve filter parameters from configuration
filter_order = config["filter"].get("order", 4)
filter_wn = config["filter"].get("wn", 0.1)
filter_type = config["filter"].get("type", "lowpass")

filtered_windows_by_label = {}

for label, windows in windowed_samples_by_label.items():
    filtered_windows = []
    print(f"Applying filter to {label} ({len(windows)} windows)")
    for i, window_df in enumerate(windows):
        # Copy to avoid modifying original data
        filtered_df = window_df.copy()
        # Apply filter to each sensor axis
        for axis in [
            "acc_x",
            "acc_y",
            "acc_z",
            "gyro_x",
            "gyro_y",
            "gyro_z",
            "acc_x_world",
            "acc_y_world",
            "acc_z_world",
        ]:
            filtered_df[axis] = utils.apply_filter(
                window_df[axis].values,
                order=filter_order,
                wn=filter_wn,
                filter_type=filter_type,
            )
        filtered_windows.append(filtered_df)
        # if (i + 1) % 50 == 0 or i == len(windows) - 1:
        #     print(f"  Processed {i + 1}/{len(windows)} windows")
    filtered_windows_by_label[label] = filtered_windows

In [ ]:
# Visualize a single filtered window from each class
for label, dfs in filtered_windows_by_label.items():
    if not dfs:
        continue
    df = dfs[0]
    n_samples = len(df)
    time_axis = np.arange(n_samples) / sampling_rate

    fig, axs = plt.subplots(3, 1, figsize=(14, 6), sharex=True)

    # Plot accelerometer signals
    axs[0].plot(time_axis, df['acc_x'], label='acc_x')
    axs[0].plot(time_axis, df['acc_y'], label='acc_y')
    axs[0].plot(time_axis, df['acc_z'], label='acc_z')
    axs[0].set_title(f'Filtered Accelerometer Signals - Class: {label}')
    axs[0].set_ylabel('Acceleration (g)')
    axs[0].legend()

    # Plot world-transformed accelerometer signals
    axs[1].plot(time_axis, df["acc_x_world"], label="acc_x_world")
    axs[1].plot(time_axis, df["acc_y_world"], label="acc_y_world")
    axs[1].plot(time_axis, df["acc_z_world"], label="acc_z_world")
    axs[1].set_title(f"Filtered World-Transformed Accelerometer Signals - Class: {label}")
    axs[1].set_ylabel("Acceleration (g)")
    axs[1].legend()

    # Plot gyroscope signals
    axs[2].plot(time_axis, df['gyro_x'], label='gyro_x')
    axs[2].plot(time_axis, df['gyro_y'], label='gyro_y')
    axs[2].plot(time_axis, df['gyro_z'], label='gyro_z')
    axs[2].set_title(f'Filtered Gyroscope Signals - Class: {label}')
    axs[2].set_xlabel('Time (seconds)')
    axs[2].set_ylabel('Angular Velocity (deg/s)')
    axs[2].legend()

    plt.tight_layout()
    plt.show()

## Preprocessing for training


#### Balance Background Class

To prevent the background class from dominating the dataset, we will:
- Calculate the total number of windows for all classes except 'background'.
- Keep only 60% of that value as the number of background windows.
- This helps balance the dataset and avoid bias toward the background class during training.


In [ ]:
# Calculate total windows for all classes except 'background'
non_background_labels = [label for label in filtered_windows_by_label if label.lower() != 'background']
total_non_background_windows = sum(len(filtered_windows_by_label[label]) for label in non_background_labels)

# Determine the number of background windows to keep (60% of total non-background windows)
max_background_windows = int(0.6 * total_non_background_windows)

# Get all background windows (if any)
background_windows = filtered_windows_by_label.get('background', [])

# Randomly select up to max_background_windows for background
if len(background_windows) > max_background_windows:
    rng = np.random.default_rng(seed=42)
    selected_indices = rng.choice(len(background_windows), size=max_background_windows, replace=False)
    background_windows_limited = [background_windows[i] for i in selected_indices]
else:
    background_windows_limited = background_windows

print(f"Total non-background windows: {total_non_background_windows}")
print(f"Keeping {len(background_windows_limited)} background windows (max allowed: {max_background_windows})")

# Optionally, update the filtered_windows_by_label dict for downstream use
filtered_windows_by_label['background'] = background_windows_limited


#### Transform DataFrames to NumPy Arrays

We'll convert the filtered windowed DataFrames into NumPy arrays for efficient machine learning processing.

- **X**: 3D NumPy array of shape (instances, time_steps, features)
- **y**: 1D NumPy array of labels for each instance

Each instance corresponds to a windowed segment of sensor data. The feature set includes world-frame accelerometer data, as well as gyroscope data:
- acc_x_world, acc_y_world, acc_z_world (world frame)
- gyro_x, gyro_y, gyro_z

This structure is required for most deep learning and classical ML models.

Example:
```python
X = np.array([
    # class1 instance
    [[acc_x_world1, acc_y_world1, acc_z_world1, gyro_x1, gyro_y1, gyro_z1], ...],
    # class2 instance
    [[acc_x_world2, acc_y_world2, acc_z_world2, gyro_x2, gyro_y2, gyro_z2], ...],
])

y = np.array([
    "g_clockwise",
    "g_counterclockwise",
])
```

In [ ]:
# Prepare lists to store all windowed instances and their labels
all_instances = []
all_labels = []

# Iterate over each activity class and its filtered DataFrames
for label, dfs in filtered_windows_by_label.items():
    for df in dfs:
        try:
            # Extract only world-frame accelerometer and gyroscope data as a NumPy array (shape: time_steps x features)
            instance = df[[
                'acc_x_world', 'acc_y_world', 'acc_z_world',
                'gyro_x', 'gyro_y', 'gyro_z'
            ]].values
            all_instances.append(instance)
            all_labels.append(label)
        except Exception as exc:
            print(f"Error for label: {label}")
            print("DataFrame columns:", df.columns.tolist())
            print("Exception:", exc)

# Convert lists to NumPy arrays for ML processing
X = np.array(all_instances)
y = np.array(all_labels)

print(f"X shape: {X.shape} (instances, time_steps, features)")
print(f"y shape: {y.shape} (instances,)")

#### Encoding

In [ ]:
# Get unique class names and their encoded values
unique_classes, encoded_values = np.unique(y, return_inverse=True)

print("Class name -> Encoded value mapping:")
for idx, class_name in enumerate(unique_classes):
    print(f"Class: {class_name}, Encoded Value: {idx}")

#### Train/Validation/Test Split

To evaluate model performance and prevent overfitting, we split the dataset into three subsets:

- **Training set**: Used to fit the model parameters.
- **Validation set**: Used for hyperparameter tuning and model selection during training.
- **Test set**: Used for final evaluation of model generalization after training is complete.

A common split is 60% training, 20% validation, and 20% test. Stratified sampling is recommended to preserve class balance across splits.

This split ensures that the model is trained, tuned, and evaluated on separate data, providing a robust assessment of its real-world performance.

In [ ]:
# Use encoded integer labels for y
y = encoded_values

# Set random seed for reproducibility
RANDOM_STATE = 42

# Stratified split to maintain class balance
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, shuffle=True, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, shuffle=True, stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train set:     {X_train.shape}, {y_train.shape}")
print(f"Validation set: {X_val.shape}, {y_val.shape}")
print(f"Test set:      {X_test.shape}, {y_test.shape}")

# Print class balance for each set
print("\nClass balance in each set:")
def print_class_balance(y, name):
    counts = Counter(y)
    sorted_counts = dict(sorted(counts.items()))
    print(f"{name}: {sorted_counts}")

print_class_balance(y_train, "Train")
print_class_balance(y_val, "Validation")
print_class_balance(y_test, "Test")


#### Feature Scaling

To ensure all features contribute equally to model training, we apply standardization (zero mean, unit variance) using `StandardScaler` from scikit-learn.

- The scaler is fit **only on the training set** to prevent data leakage.
- The same transformation is then applied to the validation and test sets.
- This is a best practice for robust and fair model evaluation.

In [ ]:
# Reshape to (instances * time_steps, features) for per-axis scaling
n_instances, n_time_steps, n_features = X_train.shape

X_train_reshaped = X_train.reshape(-1, n_features)
X_val_reshaped = X_val.reshape(-1, n_features)
X_test_reshaped = X_test.reshape(-1, n_features)

# Fit scaler on training data only (per feature/axis)
scaler = StandardScaler()
scaler.fit(X_train_reshaped)

# Transform all splits and reshape back to original shape
X_train_scaled = scaler.transform(X_train_reshaped).reshape(n_instances, n_time_steps, n_features)
X_val_scaled = scaler.transform(X_val_reshaped).reshape(X_val.shape)
X_test_scaled = scaler.transform(X_test_reshaped).reshape(X_test.shape)

print(f"Scaled X_train shape: {X_train_scaled.shape}")
print(f"Scaled X_val shape:   {X_val_scaled.shape}")
print(f"Scaled X_test shape:  {X_test_scaled.shape}")

### Visualizing Scaled vs Original Features

To ensure the scaling process worked as intended, we will:
- Plot the distribution of each feature (axis) before and after scaling
- Visualize a few time series examples for both original and scaled data
- Print summary statistics (mean, std) for each feature before and after scaling

This helps verify that each feature is now zero-mean and unit-variance, and that the time series structure is preserved.

In [ ]:
# Pick a subset of features to visualize (all features)
feature_names = [
    'acc_x_world', 'acc_y_world', 'acc_z_world',
    'gyro_x', 'gyro_y', 'gyro_z'
]

# Flatten all instances and time steps for each feature
X_train_flat_orig = X_train.reshape(-1, X_train.shape[-1])
X_train_flat_scaled = X_train_scaled.reshape(-1, X_train_scaled.shape[-1])

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(feature_names):
    ax = axes[i]
    sns.kdeplot(X_train_flat_orig[:, i], label='Original', ax=ax, fill=True, color='tab:blue', alpha=0.5)
    sns.kdeplot(X_train_flat_scaled[:, i], label='Scaled', ax=ax, fill=True, color='tab:orange', alpha=0.5)
    ax.set_title(feat)
    ax.legend()
plt.suptitle('Feature Distributions Before and After Scaling (Train Set)', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# Print summary statistics including min and max
print('Feature statistics (Train set):')
stats = []
for i, feat in enumerate(feature_names):
    orig_mean = X_train_flat_orig[:, i].mean()
    orig_std = X_train_flat_orig[:, i].std()
    orig_min = X_train_flat_orig[:, i].min()
    orig_max = X_train_flat_orig[:, i].max()
    scaled_mean = X_train_flat_scaled[:, i].mean()
    scaled_std = X_train_flat_scaled[:, i].std()
    scaled_min = X_train_flat_scaled[:, i].min()
    scaled_max = X_train_flat_scaled[:, i].max()
    stats.append([
        feat, orig_mean, orig_std, orig_min, orig_max,
        scaled_mean, scaled_std, scaled_min, scaled_max
    ])

stats_df = pd.DataFrame(
    stats,
    columns=['Feature', 'Orig Mean', 'Orig Std', 'Orig Min', 'Orig Max',
             'Scaled Mean', 'Scaled Std', 'Scaled Min', 'Scaled Max']
)
print(stats_df.round(3))

In [ ]:
# Select 2 random instances and plot all 6 features (acc_x_world, acc_y_world, acc_z_world, gyro_x, gyro_y, gyro_z)
example_indices = np.random.choice(X_train.shape[0], size=2, replace=False)
features_to_plot = list(range(6))
feature_labels = feature_names

for idx in example_indices:
    fig, axes = plt.subplots(len(features_to_plot), 1, figsize=(14, 18), sharex=True)
    for i, (f_idx, label) in enumerate(zip(features_to_plot, feature_labels)):
        ax1 = axes[i]
        color_orig = 'tab:blue'
        color_scaled = 'tab:orange'
        ln1 = ax1.plot(X_train[idx, :, f_idx], label='Original', color=color_orig, linestyle='-')
        ax1.set_ylabel(f'{label}\n(Original)', color=color_orig)
        ax1.tick_params(axis='y', labelcolor=color_orig)
        ax1.axhline(0, color='gray', linewidth=0.5, linestyle=':')
        ax2 = ax1.twinx()
        ln2 = ax2.plot(X_train_scaled[idx, :, f_idx], label='Scaled', color=color_scaled, linestyle='--')
        ax2.set_ylabel(f'{label}\n(Scaled)', color=color_scaled)
        ax2.tick_params(axis='y', labelcolor=color_scaled)
        lns = ln1 + ln2
        labels_ = [l.get_label() for l in lns]
        ax1.legend(lns, labels_, loc='upper right')
    axes[0].set_title(
        f'Time Series Example (Instance {idx}) - Label: {y_train[idx]}\n'
        'Original (left axis) vs Scaled (right axis)')
    axes[-1].set_xlabel('Time Step')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

## Classifier - Neural Network

### Prepare Neural Network Input Shape and Output Classes

Before building the neural network, we need to:
- Determine the input shape for the model (number of time steps and features per instance)
- Calculate the number of output classes for classification

This ensures the model architecture matches the data dimensions and the classification task.


In [ ]:
input_data_shape = X_train.shape[1:]
print(f'Input shape for the model: {input_data_shape}')

y_np_array = np.array(y)
n_outputs = len(np.unique(y_np_array))
print(f'Number of output classes: {n_outputs}')

### Create the Neural Network (NN) Architecture and instantiate the model
**We designed a hybrid CNN-LSTM architecture optimized for wearable sensor data:**  

1. **Input Layer (120×6)**  
   Processes 1.2s windows of 6-axis IMU data (3 accel + 3 gyro) at 100Hz, ensuring complete gesture coverage.

2. **Convolutional Layers**  
   - `Conv1D(64,k5)`: Detects 50ms local patterns (acceleration spikes, rotation signatures)  
   - `Conv1D(128,k3)`: Learns compound 180ms motion features  
   - `MaxPooling`: Downsamples while preserving key features  

3. **Bidirectional LSTM**  
   Processes sequences forward/backward to capture reversal dynamics (e.g., "U" gestures), maintaining temporal context.

4. **Final LSTM (32 units)**  
   Compresses sequence into a single context vector representing the entire gesture.

5. **Dense + Output Layers**  
   Classifies gestures using distilled features with dropout regularization.

*This structure extracts spatial features (CNN) and temporal dependencies (LSTM), validated to achieve >90% accuracy across motion contexts [5,6,7].*

In [ ]:
# Simple CNN model
# model = Sequential(
#     [
#         Input(shape=(120, 6)),
#         Conv1D(32, 5, activation="relu"),
#         MaxPooling1D(2),
#         Conv1D(64, 3, activation="relu"),
#         GlobalAveragePooling1D(),
#         Dense(9, activation="softmax"),
#     ]
# )

# More complex CNN-LSTM model
model = Sequential(
    [
        Input(shape=(window_size, 6)),
        Conv1D(64, kernel_size=5, activation="relu", padding="same"),
        MaxPooling1D(pool_size=2),
        Conv1D(128, kernel_size=3, activation="relu", padding="same"),
        MaxPooling1D(pool_size=2),
        Bidirectional(LSTM(64, return_sequences=True)),
        LSTM(32),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(9, activation="softmax"),  # [batch, 9] (8 gestures + background)
    ]
)

# Alternative model with regularization and dropout
# model = Sequential(
#     [
#         Input(shape=(window_size, 6)),
#         GaussianNoise(0.02),  # Input-level regularization
#         Conv1D(64, 5, activation="relu", padding="same"),
#         BatchNormalization(),
#         MaxPooling1D(2),
#         Dropout(0.3),  # Additional spatial dropout
#         Conv1D(64, 3, activation="relu", padding="same"),  # Reduced filters
#         BatchNormalization(),
#         MaxPooling1D(2),
#         Dropout(0.3),
#         # Simplified temporal modeling
#         Bidirectional(
#             LSTM(64, return_sequences=False, dropout=0.3)
#         ),  # Single LSTM
#         Dropout(0.4),
#         Dense(64, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
#         Dropout(0.5),
#         Dense(9, activation="softmax"),
#     ]
# )



model.add(Dense(n_outputs, activation="softmax"))

Plot the Architecture of ot the TensorFlow model

In [ ]:
from tensorflow.keras.utils import plot_model
from IPython.display import Image

plot_model(
    model,
    to_file="img/model.png",
    show_shapes=True,
    show_layer_names=True,
    show_dtype=True,
    show_layer_activations=True,
    expand_nested=True,
    dpi=300,
)
display(Image(filename="img/model.png"))

Plot the summary of the TensorFlow model

In [ ]:
model.summary()

### Build the NN model

In [ ]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

# # Slower learning rate + momentum
# optimizer = tf.keras.optimizers.SGD(
#     learning_rate=0.01, 
#     momentum=0.9
# )

# # Add gradient clipping
# model.compile(
#     loss="sparse_categorical_crossentropy",
#     optimizer=optimizer,
#     metrics=["accuracy"]
# )

### Calculate class weights for imbalanced datasets

In [ ]:
# Use sqrt weighting for less extreme balancing
class_weights = np.sqrt(
    class_weight.compute_class_weight(
        "balanced", classes=np.unique(y_train), y=y_train
    )
)

class_weights_dict = dict(enumerate(class_weights))

print("Class weights:")
for i, weight in enumerate(class_weights_dict.values()):
    print(f"Class {i}: {weight:.2f}")

### Train the NN model

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=30,
    batch_size=32,
    class_weight=class_weights_dict,
    validation_data=(X_val_scaled, y_val),
    # callbacks=[
    #     tf.keras.callbacks.EarlyStopping(
    #         patience=10, restore_best_weights=True
    #     ),
    #     tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
    # ],
    # callbacks=[
    #     tf.keras.callbacks.EarlyStopping(
    #         patience=5,
    #         monitor="val_accuracy",
    #         mode="max",
    #         restore_best_weights=True,
    #     )
    # ],
    callbacks=[
        EarlyStopping(
            patience=15,
            monitor="accuracy",
            mode="max",
            restore_best_weights=True,
        ),
        ReduceLROnPlateau(factor=0.2, patience=10, min_lr=1e-6),
        ModelCheckpoint("best_model_cnn.h5", save_best_only=True),
    ],
)

### Evaluate the model on the test data

In [ ]:
test_loss, test_acc = model.evaluate(
    X_test_scaled, y_test
)
print(f"Test Accuracy: {test_acc:.4f}")

# Confusion matrix
y_pred = model.predict(X_test_scaled)
y_pred_classes = np.argmax(y_pred, axis=1)
conf_matrix = confusion_matrix(y_test, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=list(unique_classes),
    yticklabels=list(unique_classes),
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# Classification report
print(
    classification_report(y_test, y_pred_classes, target_names=unique_classes)
)

### Plot and interpret the learning curves: Loss and Accuracy based on the training and validation sets

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Accuracy Curves")
plt.ylabel("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Loss Curves")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.legend()
plt.tight_layout()
plt.show()

## Random Forest Classifier

In [ ]:
# Flatten the 3D arrays to 2D for Random Forest
n_train, t_train, f_train = X_train_scaled.shape
n_val, t_val, f_val = X_val_scaled.shape
n_test, t_test, f_test = X_test_scaled.shape

X_train_rf = X_train_scaled.reshape(n_train, t_train * f_train)
X_val_rf = X_val_scaled.reshape(n_val, t_val * f_val)
X_test_rf = X_test_scaled.reshape(n_test, t_test * f_test)

from sklearn.ensemble import RandomForestClassifier

# Instantiate the classifier
rf_clf = RandomForestClassifier(
    n_estimators=100, 
    random_state=42, 
    class_weight='balanced', 
    n_jobs=-1
)

# Train the classifier
rf_clf.fit(X_train_rf, y_train)

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Predict on the test set
y_pred_rf = rf_clf.predict(X_test_rf)

# Accuracy
test_acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Test Accuracy: {test_acc_rf:.4f}")

# Confusion matrix
conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(10, 8))
sns.heatmap(
    conf_matrix_rf,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=list(unique_classes),
    yticklabels=list(unique_classes),
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Random Forest Confusion Matrix")
plt.show()

# Classification report
print(classification_report(y_test, y_pred_rf, target_names=unique_classes))